# BRIDGET: compas test



## Dataset Preprocessing


### Libraries, Retrieving data

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json
import yaml
import joblib
import random
import time
import functools
import pickle
import re
import orjson
import alibi
import ignite
import copy

from IPython import display
from itertools import combinations, product
from tqdm import tqdm
from matplotlib import pyplot as plt
from collections import Counter

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optimizer


from river import rules, tree, datasets, drift, metrics, evaluate
from river import imblearn
from river import preprocessing
from river import optim
from river import metrics
from river import feature_extraction, feature_selection
from river import ensemble, linear_model, forest, compose

from torchsummary import summary
from torch.utils.data import TensorDataset, DataLoader

from ignite.metrics import Accuracy, Loss
from ignite.engine import Engine, Events, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import EarlyStopping, ModelCheckpoint
from ignite.contrib.handlers import global_step_from_engine
from sklearn.preprocessing import MinMaxScaler
from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

from alibi.explainers.cfproto import CounterFactualProto

from bridget_utils import *
from orchestrator import *
from master_config import *
from classes import BetaUser, DeferralNet, PyTorchWrapper, RiverModelWrapper
from bridget_main import BRIDGET, HiC, MiC


from baselines.compare_confidence import *
from baselines.differentiable_triage import *
from baselines.lce_surrogate import *
from baselines.mix_of_exps import *
from baselines.one_v_all import *
from baselines.selective_prediction import *


c:\Users\virgm\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


26-Mar-24 15:29:31 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [3]:
set_all_seeds(42)

In [4]:
ds_name= 'compas'

In [5]:
data= pd.read_csv(fr".\datasets\{ds_name}.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14788 entries, 0 to 14787
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sex              14788 non-null  object
 1   age              14788 non-null  int64 
 2   race             14788 non-null  object
 3   juv_fel_count    14788 non-null  int64 
 4   juv_misd_count   14788 non-null  int64 
 5   juv_other_count  14788 non-null  int64 
 6   priors_count     14788 non-null  int64 
 7   c_charge_degree  14788 non-null  object
 8   compas_score     14788 non-null  int64 
 9   did_recid        14788 non-null  int64 
dtypes: int64(7), object(3)
memory usage: 1.1+ MB


In [6]:
for c in data:
    print(data[c].value_counts().sum)

<bound method Series.sum of sex
Male      12093
Female     2695
Name: count, dtype: int64>
<bound method Series.sum of age
22    780
21    762
26    741
24    738
25    724
     ... 
96      2
78      1
83      1
80      1
79      1
Name: count, Length: 65, dtype: int64>
<bound method Series.sum of race
African-American    8004
Caucasian           4848
Hispanic            1083
Other                763
Asian                 57
Native American       33
Name: count, dtype: int64>
<bound method Series.sum of juv_fel_count
0     14176
1       390
2       135
3        46
4        19
5         9
8         4
10        4
6         3
20        1
13        1
Name: count, dtype: int64>
<bound method Series.sum of juv_misd_count
0     13854
1       643
2       174
3        57
4        23
8        11
6         9
5         8
12        4
7         3
13        2
Name: count, dtype: int64>
<bound method Series.sum of juv_other_count
0     13494
1       902
2       251
3        81
4        38
5         7

### Preprocessing Pipeline
1. Drop duplicates

2. Map 'sex', 'race', 'c_charge_degree'


In [7]:
data= clean_compas(data, 'compas_score', col_to_strip= 'c_charge_degree', drop_duplicates=True)

In [8]:
data.info()
data.head(n=5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5417 entries, 0 to 5416
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sex              5417 non-null   object
 1   age              5417 non-null   int64 
 2   race             5417 non-null   object
 3   juv_fel_count    5417 non-null   int64 
 4   juv_misd_count   5417 non-null   int64 
 5   juv_other_count  5417 non-null   int64 
 6   priors_count     5417 non-null   int64 
 7   c_charge_degree  5417 non-null   object
 8   did_recid        5417 non-null   int64 
dtypes: int64(6), object(3)
memory usage: 381.0+ KB


,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,did_recid
0,Female,19,African-American,0,0,0,0,F3,0
1,Female,19,African-American,0,0,0,0,M2,0
2,Female,19,African-American,0,0,0,0,M2,1
3,Female,19,African-American,0,0,0,1,F2,1
4,Female,19,African-American,0,0,0,1,F3,1


In [9]:
mapping= { 'sex': {'Female':0, 'Male':1},
           
            'race': {'Native American': 0,
            'Asian': 1,
            'Other': 2,
            'Hispanic': 3,
            'Caucasian': 4,
            'African-American': 5  },

            'c_charge_degree': {
                'X': 0, 'TCX': 0, 'NI0': 0, 'MO3': 0, 'CO3': 0, 

                'M2': 1,   
                'M1': 2,  
                'F5': 3, 'F6': 3,'F3': 3,                    
                                    
                'F7': 4, 'F2': 4, # apparently lvl 7 felonies can get u up to 15 years in prison                                                  
                'F1': 5
                }

}

data= apply_map(data, mapping)


### Splitting and Transforming data

1. Apply stratified sampling

2. Get pre-training/HiC/calibration/Mic data

3. Apply scaler

4. Get X, y

In [10]:
# Qui definiamo i vari split dei flussi 
set_all_seeds(42)
data = data.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle iniziale

class_0 = data[data['did_recid'] == 0]
class_1= data[data['did_recid'] == 1]

#  split ufficiale

splits= {
    'calibration': (0.6, 0.8),
    'mic': (0.8, 1.0),
    'avv_train': (0.0, 0.07),
    'avv_test': (0.07, 0.1),
    'hic_train': (0.1, 0.5),
    'hic_test': (0.5, 0.6)
}

dfs= {}

for name, (start, end) in splits.items():
    dfs[name]= stratif(start, end, class_0, class_1)



In [11]:
for name, df in dfs.items():
    print(f"{name} length: {len(df)}")


calibration length: 1083
mic length: 1085
avv_train length: 378
avv_test length: 163
hic_train length: 2167
hic_test length: 541


In [12]:
target= 'did_recid'

categoricals= ['sex', 'race', 'c_charge_degree']
numericals= [c for c in data if c not in categoricals and c != target]


prepr_transf = (
    (compose.Select(*numericals, *categoricals) | preprocessing.MinMaxScaler())
)

In [13]:
## ora divisione in x e y

set_all_seeds(42)

# avviamento 
X_avv_train, y_avv_train = x_y_split(dfs['avv_train'], target)
X_avv_test, y_avv_test = x_y_split(dfs['avv_test'], target)


# hic
X_hic_train, y_hic_train = x_y_split(dfs['hic_train'], target)
X_hic_test, y_hic_test = x_y_split(dfs['hic_test'], target)

# validation
X_val, y_val = x_y_split(dfs['calibration'], target)

# mic
X_mic, y_mic = x_y_split(dfs['mic'], target)



## Calibration Phase: Experts and Incremental Model Selection

### Calibrating Incremental Model

The incremental model to be chosen for Bridget is trained on the X_avv, y_avv portion of the dataset,then evaluated on the X_avv_test and y_avv_test

The calibration phase starts by assessing the results of the learning for several configurations:

    - HoeffdingTreeClassifier

    - ExtremelyFastDecisionTreeClassifier

    - AdaBoostClassifier            (base= SGTClassifier)

    - AdwinBaggingClassifier        (base= SGTClassifier)

    - SRPClassifier                 (base= SGTClassifier)

    - AdaptiveRandomForestClassifier


The metrics observed are the Accuracy, the F1Score and the Counters for the classes

In [14]:
# since all River models work with dicts, lets first transform the dfs to dict
set_all_seeds(42)
X_avv_dict= X_avv_train.to_dict(orient='records')
X_avv_dict_test= X_avv_test.to_dict(orient='records')

df_batch_1 = (dfs['hic_train']).reset_index(drop=True)
mic_data= dfs['mic'].reset_index(drop=True)
df_avv= pd.concat([dfs['avv_train'], dfs['avv_test']]).reset_index(drop=True)

test_batch_1= X_hic_test.copy()
test_batch_1[target]= y_hic_test


# setting the init params required by HIC class
RULE = False
PAST = True
SKEPT = True
GROUP = True
EVA=    True
N_BINS = 10
N_VAR = 3
MAX = 5

rule_att = 'priors_count' #random rule
rule_value = -0.7

protected= ['race', 'age', 'sex']


In [15]:
# then the models are instantiated and trained by the HiC.train function
# the HiC object is initialized by passing a random user model, its not relevant since it won't interact with the IL anyways

set_all_seeds(42)

base = tree.HoeffdingTreeClassifier(grace_period=100)

htree= tree.HoeffdingAdaptiveTreeClassifier(grace_period= 100, seed= 42)
efdt= tree.ExtremelyFastDecisionTreeClassifier(grace_period=100)
ada= ensemble.AdaBoostClassifier(model= base, n_models= 15, seed= 42)  
adwin= ensemble.ADWINBaggingClassifier(model= base, n_models= 15, seed= 42)
srp= ensemble.SRPClassifier(model= base, n_models=15, seed= 42)
arf= forest.ARFClassifier(n_models= 15, grace_period= 100, max_features='sqrt', seed=42)

models= {
    'HoeffdingATC': htree, 
    "EFDT": efdt, 
    "AdaBoostCl":ada, 
    "ADWINBAGGING": adwin, 
    "SRPCL": srp, 
    "ARF":arf   
    }

prepr_dir= DATASETS[ds_name]['base_obj_paths']['preprocessors']
os.makedirs(prepr_dir, exist_ok=True)

model_dir= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
os.makedirs(model_dir, exist_ok=True)

for model_name, model_obj in models.items():
    
    bridget_inst= HiC(RULE, PAST, SKEPT, GROUP, EVA, N_BINS, N_VAR, MAX, 
                rule_att, rule_value, 'placeholder', model_obj,
                start_performance= DATASETS[ds_name]['start_performance'],
                allocated_budget= DATASETS[ds_name]['allocated_budget'][0],
                skepticism_threshold= DATASETS[ds_name]['skepticism_threshold'],
                performance_delta= DATASETS[ds_name]['performance_delta'],
                dataset_name= ds_name,
                user_name= 'placeholder',
                batch1=df_batch_1, batch3=mic_data, batch1_test=test_batch_1, 
                target=target, 
                user_model='placeholder', 
                protected=protected, cats=categoricals, num=numericals,
                preprocessor=prepr_transf,
                training_iter= 0 
                )
    
    bridget_inst.train(X_avv_dict, y_avv_train, X_avv_dict_test, y_avv_test)

    base_preprocessor = bridget_inst.preprocessor
    base_model = bridget_inst.hic_model
    

    prepr_filename = f"{model_name}_preprocessor.pkl"
    model_filename = f"{model_name}_model.pkl"

    joblib.dump(base_preprocessor, os.path.join(prepr_dir, prepr_filename))
    joblib.dump(base_model, os.path.join(model_dir, model_filename))


trained_arf= arf


Accuracy: 65.03%
F1: 42.42%
Distribution of predictions: Counter({0: 131, 1: 32})
HoeffdingAdaptiveTreeClassifier trained
Accuracy: 58.90%
F1: 0.00%
Distribution of predictions: Counter({0: 163})
ExtremelyFastDecisionTreeClassifier trained
Accuracy: 66.87%
F1: 46.00%
Distribution of predictions: Counter({0: 130, 1: 33})
AdaBoostClassifier(HoeffdingTreeClassifier) trained
Accuracy: 66.26%
F1: 43.30%
Distribution of predictions: Counter({0: 133, 1: 30})
ADWINBaggingClassifier(HoeffdingTreeClassifier) trained
Accuracy: 65.03%
F1: 41.24%
Distribution of predictions: Counter({0: 133, 1: 30})
SRPClassifier(HoeffdingTreeClassifier) trained
Accuracy: 64.42%
F1: 40.82%
Distribution of predictions: Counter({0: 132, 1: 31})
ARFClassifier trained


### Calibrating Experts


In [16]:
with open(fr".\experts_{ds_name}.yaml", "r") as f:
    config= yaml.safe_load(f)


params_dict= config['experts']['groups']['w_dict']


In [17]:
set_all_seeds(42)
X_exp= X_hic_train.to_dict(orient='records')

X_exp_scaled= []

for x in X_exp:
    X_exp_scaled.append(prepr_transf.transform_one(x))

X_exp_final = pd.DataFrame(X_exp_scaled)

In [18]:
set_all_seeds(42)
experts_obj= {}

expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

for name in expert_names:
    expert_type= config['experts']['groups'][name]

    experts_obj[name]= BetaUser(
        belief_level= expert_type['belief_value'],
        rethink_level= 0.8, # as suggested by the og FRANK implementation
        fairness= True,
        fpr= expert_type['target_FPR'],
        fnr= expert_type['target_FNR'],
        alpha= 0.9,
        features_dict= params_dict,
        seed= expert_type['group_seed']
        )
    res = experts_obj[name].fit(X_exp_final, y_hic_train, tol= 0.001)

    save_dir= fr".\trained_experts\{ds_name}"
    os.makedirs(save_dir, exist_ok= True)
    file_path = os.path.join(save_dir, f"{name}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(experts_obj[name], f)
    print(f"Expert saved in: {file_path}")

    
    print(f"{'='*30}")
    print(f" EXPERT CALIBRATION REPORT ")
    print(f"{'='*30}")

    print(f"\n[EXPERT: {name}]")
    print(f"\n[FALSE POSITIVE RATE]")
    print(f"  - Iters:      {res['fpr iters number']}")
    print(f"  - Beta:       {res['calibrated_fpr_beta']:.4f}")
    print(f"  - Target:     {res['target_fpr']}")
    print(f"  - Achieved:   {res['achieved_fpr']:.4f}")

    print(f"\n[FALSE NEGATIVE RATE]")
    print(f"  - Iters:      {res['fnr iters number']}")
    print(f"  - Beta:       {res['calibrated_fnr_beta']:.4f}")
    print(f"  - Target:     {res['target_fnr']}")
    print(f"  - Achieved:   {res['achieved_fnr']:.4f}")
    


Expert saved in: .\trained_experts\compas\accurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      11
  - Beta:       -3.2227
  - Target:     0.05
  - Achieved:   0.0498

[FALSE NEGATIVE RATE]
  - Iters:      13
  - Beta:       -2.6123
  - Target:     0.05
  - Achieved:   0.0501
Expert saved in: .\trained_experts\compas\accurate_not_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_not_trusting]

[FALSE POSITIVE RATE]
  - Iters:      12
  - Beta:       -3.0762
  - Target:     0.05
  - Achieved:   0.0491

[FALSE NEGATIVE RATE]
  - Iters:      12
  - Beta:       -2.7832
  - Target:     0.05
  - Achieved:   0.0506
Expert saved in: .\trained_experts\compas\inaccurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: inaccurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      14
  - Beta:       -1.2329
  - Target:     0.3
  - Achieved:   0.3008

[FALSE NEGATIVE RATE]
  - Iters:      14
  - Beta:       -0.3784
  - Target

## BRIDGET decision making



In [19]:
# base params for all users
initial_ordering= [c for c in data if c != target]

# retrieve preprocessor and incremental learner
rules= FRANK_RULES


base_params= {
    "cats": categoricals, #lst
    "num": numericals, #lss
    "warm_up_set": df_avv.copy(),
    "batch1": df_batch_1.copy(),
    "batch1_test": test_batch_1.copy(),
    "batch3":dfs['mic'].copy(),
    "validation_set": dfs['calibration'].copy(), #obtained by hic
    "feature_order": initial_ordering.copy(), #without the label ?
    "incremental_learner_name":"ARF",
    "n_cols": len(initial_ordering), #obtained by hic
    "mic_model_name": "Def_Net",
    "strat_1_name": "Confidence",
    "strat_2_name": "Mao",
    
}


learner_path= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
path= os.path.join(learner_path, f"{base_params["incremental_learner_name"]}_model.pkl")
trained_learner= joblib.load(path)

prepr_path= DATASETS[ds_name]['base_obj_paths']['preprocessors']
path= os.path.join(prepr_path, f"{base_params["incremental_learner_name"]}_preprocessor.pkl")
base_prepr= joblib.load(path)

base_objects = {
      "preprocessor": copy.deepcopy(base_prepr), #or take from path
      "incremental_learner":copy.deepcopy(trained_learner), # or take from path
      "scaler": None
} 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#add these once you get to mic phase: "run_confidence": True, "run_mao": True

### Expert: Accurate, Trusting 

#### HIC

In [20]:
#setting up fixed params

current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [21]:
run_hic(ds_name, params, objects)

 34%|███▎      | 726/2167 [01:14<02:42,  8.86it/s]

2026-03-19 12:38:37,324 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▎      | 726/2167 [01:14<02:27,  9.78it/s]


0.9450261780104712 1.061764705882353


2026-03-19 12:38:40,415 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

2026-03-19 12:41:48,307 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▍      | 743/2167 [01:37<03:06,  7.63it/s]

0.9823369565217391 1.0183098591549296
Training Results - Epoch[1] Avg accuracy: 0.49 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.49 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 2: Learning Rate 0.001



2026-03-19 12:41:48,573 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[3] Avg accuracy: 0.49 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 4: Learning Rate 0.001
0.9823369565217391 1.0183098591549296
Training Results - Epoch[1] Avg accuracy: 0.49 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.50 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.52 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.47 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.54 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.69
End of Epoch 4: Learning R

#### CALIB

In [22]:
#chosen nets: "large", 32/16

net_layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']

# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{net_layers[0]}_{net_layers[1]}"
                          )

    pattern = f"{net_layers[0]}_{net_layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [20]:
current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2,
                            min_coverage=0.6)
    
  




 25%|██▍       | 270/1085 [00:42<01:50,  7.37it/s]

2026-03-24 15:35:54,358 | WARNING | Drift here! Record index 270
2026-03-24 15:35:54,376 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:42<01:51,  7.30it/s]

2026-03-24 15:36:36,966 | WARNING | Drift here! Record index 270
2026-03-24 15:36:37,001 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:42<01:50,  7.41it/s]

...

 25%|██▍       | 270/1085 [00:41<02:17,  5.92it/s]

2026-03-24 15:44:19,770 | WARNING | Drift here! Record index 270
2026-03-24 15:44:19,789 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:41<02:17,  5.91it/s]

2026-03-24 15:45:01,549 | WARNING | Drift here! Record index 270
2026-03-24 15:45:01,568 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


100%|██████████| 1085/1085 [02:34<00:00,  7.03it/s]


### Expert: Inaccurate, Trusting 

#### HIC

In [24]:
#setting up fixed params

current_user_name= "inaccurate_trusting"
user_suffix= 'inacc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [25]:
run_hic(ds_name, params, objects)
       

 34%|███▍      | 742/2167 [01:19<02:55,  8.12it/s]

2026-03-19 12:43:19,512 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▍      | 742/2167 [01:19<02:33,  9.31it/s]

0.9450261780104712 1.061764705882353
Training Results - Epoch[1] Avg accuracy: 0.47 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.47 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.47 Avg loss: 0.70


Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.47 Avg loss: 0.70


...

0.9341085271317829 1.0758928571428572
Training Results - Epoch[1] Avg accuracy: 0.46 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.46 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 2: Learning Rate 0.001



2026-03-19 12:46:17,281 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[3] Avg accuracy: 0.46 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.46 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 4: Learning Rate 0.001
0.9341085271317829 1.0758928571428572
Training Results - Epoch[1] Avg accuracy: 0.46 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.47 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.50 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.69
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.50 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.69
End of Epoch 4: Learning R

2026-03-19 12:46:18,525 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[10] Avg accuracy: 0.63 Avg loss: 0.68
Validation Results - Epoch[10] Avg accuracy: 0.63 Avg loss: 0.68
End of Epoch 10: Learning Rate 0.0008
Training Results - Epoch[11] Avg accuracy: 0.64 Avg loss: 0.68
Validation Results - Epoch[11] Avg accuracy: 0.64 Avg loss: 0.68
End of Epoch 11: Learning Rate 0.0008
Training Results - Epoch[12] Avg accuracy: 0.65 Avg loss: 0.68
Validation Results - Epoch[12] Avg accuracy: 0.64 Avg loss: 0.68
End of Epoch 12: Learning Rate 0.0008
Training Results - Epoch[13] Avg accuracy: 0.66 Avg loss: 0.67
Validation Results - Epoch[13] Avg accuracy: 0.63 Avg loss: 0.68
End of Epoch 13: Learning Rate 0.0008
Training Results - Epoch[14] Avg accuracy: 0.67 Avg loss: 0.67
Validation Results - Epoch[14] Avg accuracy: 0.63 Avg loss: 0.67
End of Epoch 14: Learning Rate 0.0008


#### Calib

In [26]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']
# retrieving df and best net path

for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [21]:
current_user_name= "inaccurate_trusting"
user_suffix= 'inacc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2,
                            min_coverage=0.6)
    
  

 25%|██▍       | 270/1085 [00:38<02:02,  6.66it/s]

2026-03-24 16:20:46,362 | WARNING | Drift here! Record index 270
2026-03-24 16:20:46,377 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:38<01:59,  6.80it/s]

2026-03-24 16:21:24,814 | WARNING | Drift here! Record index 270
2026-03-24 16:21:24,839 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:38<02:11,  6.19it/s]

...

 25%|██▍       | 270/1085 [00:38<02:21,  5.75it/s]

2026-03-24 16:41:43,078 | WARNING | Drift here! Record index 270
2026-03-24 16:41:43,097 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:37<02:22,  5.71it/s]

2026-03-24 16:47:33,766 | WARNING | Drift here! Record index 270
2026-03-24 16:47:33,784 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


100%|██████████| 1085/1085 [02:36<00:00,  6.91it/s]


### Expert: Accurate, Not Trusting 

#### HIC

In [27]:
#setting up fixed params

current_user_name= "accurate_not_trusting"
user_suffix= 'acc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [28]:
run_hic(ds_name, params, objects)

 34%|███▍      | 742/2167 [01:18<02:49,  8.41it/s]

2026-03-19 12:47:46,760 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▍      | 742/2167 [01:18<02:30,  9.47it/s]

0.9116161616161617 1.107361963190184
Training Results - Epoch[1] Avg accuracy: 0.45 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.45 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001



2026-03-19 12:47:47,019 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

2026-03-19 12:50:54,133 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▍      | 743/2167 [01:33<03:00,  7.91it/s]

0.938961038961039 1.069526627218935
Training Results - Epoch[1] Avg accuracy: 0.47 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.47 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.47 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001



2026-03-19 12:50:54,365 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[4] Avg accuracy: 0.47 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 4: Learning Rate 0.001
0.938961038961039 1.069526627218935
Training Results - Epoch[1] Avg accuracy: 0.47 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.47 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.48 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.42 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.50 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.48 Avg loss: 0.69
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.51 Avg loss: 0.69
Validation Results - Epoch[5] Avg accuracy: 0.52 Avg loss: 0.69
End of Epoch 5: Learning Rat

#### CALIB

In [29]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']
# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [22]:
current_user_name= "accurate_not_trusting"
user_suffix= 'acc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2,
                            min_coverage=0.6)
    
  

 25%|██▍       | 270/1085 [00:39<02:14,  6.07it/s]

2026-03-24 16:53:26,000 | WARNING | Drift here! Record index 270
2026-03-24 16:53:26,019 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:40<01:45,  7.69it/s]

2026-03-24 17:04:32,927 | WARNING | Drift here! Record index 270
2026-03-24 17:04:32,939 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Confidence


 25%|██▍       | 270/1085 [00:40<01:44,  7.79it/s]

...

 25%|██▍       | 270/1085 [00:38<01:56,  6.97it/s]

2026-03-24 17:15:00,740 | WARNING | Drift here! Record index 270
2026-03-24 17:15:00,757 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:38<01:57,  6.93it/s]

2026-03-24 17:23:11,639 | WARNING | Drift here! Record index 270
2026-03-24 17:23:11,656 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:38<01:56,  6.99it/s]


### Expert: Inaccurate, Not Trusting 

#### HIC

In [30]:
#setting up fixed params

current_user_name= "inaccurate_not_trusting"
user_suffix= 'inacc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [31]:
run_hic(ds_name, params, objects)

 34%|███▍      | 742/2167 [01:17<02:41,  8.84it/s]

2026-03-19 12:52:23,436 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▍      | 742/2167 [01:18<02:30,  9.49it/s]

0.907035175879397 1.1141975308641976
Training Results - Epoch[1] Avg accuracy: 0.45 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.45 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001



2026-03-19 12:52:23,661 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

2026-03-19 12:55:29,764 | WARNING | DRIFT DETECTED | Phase: HiC | Strat: None


 34%|███▍      | 743/2167 [01:32<02:58,  7.99it/s]

0.9082914572864321 1.1123076923076922
Training Results - Epoch[1] Avg accuracy: 0.45 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.45 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.70



2026-03-19 12:55:30,021 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.45 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 4: Learning Rate 0.001
0.9082914572864321 1.1123076923076922
Training Results - Epoch[1] Avg accuracy: 0.45 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.45 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.42 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.47 Avg loss: 0.69
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.50 Avg l

#### CALIB

In [ ]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']

# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [23]:
current_user_name= "inaccurate_not_trusting"
user_suffix= 'inacc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2,
                            min_coverage=0.6)
    
  




 25%|██▍       | 270/1085 [00:43<01:56,  6.98it/s]

2026-03-24 17:26:41,707 | WARNING | Drift here! Record index 270
2026-03-24 17:26:41,724 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:42<01:57,  6.93it/s]

2026-03-24 17:27:24,997 | WARNING | Drift here! Record index 270
2026-03-24 17:27:25,011 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:43<01:56,  6.97it/s]

...

 25%|██▍       | 270/1085 [00:40<02:02,  6.66it/s]

2026-03-24 17:47:11,075 | WARNING | Drift here! Record index 270
2026-03-24 17:47:11,092 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:40<02:03,  6.62it/s]

2026-03-24 17:53:05,687 | WARNING | Drift here! Record index 270
2026-03-24 17:53:05,738 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:41<02:04,  6.55it/s]


## Benchmarking 

since COMPAS has their score already, lets test whether our experts are way off their prediction

### Human in Command

### Machine in Command (anche qui la 32/16 palesemente)

#### Experts re-calibration and prediction

In [14]:
# a note: in the BRIDGET pipeline, the Machine in Command phase receives the information provided by a scaler from the River lib

# however, since the ablation study will focus on a neural network that has to learn on the Human-in-Command portion anyways, 
# it needs its own dedicated scaler (a standard sklearn Min Max Scaler)

# now do the fitting again with the Min Max scaler, then get the predictions using the baseline func

set_all_seeds(42)

with open(fr".\experts_{ds_name}.yaml", "r") as f:
    config= yaml.safe_load(f)


params_dict= config['experts']['groups']['w_dict']

scaler= MinMaxScaler()
experts_obj= {}

X_scaled= scaler.fit_transform(X_hic_train)
X_scaled = pd.DataFrame(X_scaled, columns= X_hic_train.columns)

expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

for name in expert_names:
    expert_type= config['experts']['groups'][name]

    experts_obj[name]= BetaUser(
        belief_level= expert_type['belief_value'],
        rethink_level= 0.8, # as suggested by the og FRANK implementation
        fairness= True,
        fpr= expert_type['target_FPR'],
        fnr= expert_type['target_FNR'],
        alpha= 0.9,
        features_dict= params_dict,
        seed= expert_type['group_seed']
        )
    res = experts_obj[name].fit(X_scaled, y_hic_train, tol= 0.001)

    save_dir= fr".\baseline\experts\{ds_name}"
    os.makedirs(save_dir, exist_ok= True)
    file_path = os.path.join(save_dir, f"{name}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(experts_obj[name], f)
    print(f"Expert saved in: {file_path}")

    
    print(f"{'='*30}")
    print(f" EXPERT CALIBRATION REPORT ")
    print(f"{'='*30}")

    print(f"\n[EXPERT: {name}]")
    print(f"\n[FALSE POSITIVE RATE]")
    print(f"  - Iters:      {res['fpr iters number']}")
    print(f"  - Beta:       {res['calibrated_fpr_beta']:.4f}")
    print(f"  - Target:     {res['target_fpr']}")
    print(f"  - Achieved:   {res['achieved_fpr']:.4f}")

    print(f"\n[FALSE NEGATIVE RATE]")
    print(f"  - Iters:      {res['fnr iters number']}")
    print(f"  - Beta:       {res['calibrated_fnr_beta']:.4f}")
    print(f"  - Target:     {res['target_fnr']}")
    print(f"  - Achieved:   {res['achieved_fnr']:.4f}")
    


Expert saved in: .\baseline\experts\compas\accurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      12
  - Beta:       -3.1738
  - Target:     0.05
  - Achieved:   0.0509

[FALSE NEGATIVE RATE]
  - Iters:      11
  - Beta:       -2.6367
  - Target:     0.05
  - Achieved:   0.0504
Expert saved in: .\baseline\experts\compas\accurate_not_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_not_trusting]

[FALSE POSITIVE RATE]
  - Iters:      11
  - Beta:       -3.0273
  - Target:     0.05
  - Achieved:   0.0497

[FALSE NEGATIVE RATE]
  - Iters:      11
  - Beta:       -2.8320
  - Target:     0.05
  - Achieved:   0.0502
Expert saved in: .\baseline\experts\compas\inaccurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: inaccurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      14
  - Beta:       -1.2085
  - Target:     0.3
  - Achieved:   0.2999

[FALSE NEGATIVE RATE]
  - Iters:      14
  - Beta:       -0.4272
  - Tar

In [ ]:
## since each strategy requires the label produced by the users, lets produce them :) 

## we'll use the calibrated experts in the (trained experts) folder 
# (alternatively, since you gotta rerun the preprocessing and calibration, just use the experts obj dict)
# to make it more proper, lets just retrieve the paths

experts_obj= {}
scaler= MinMaxScaler()
expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

res= {}

for name in expert_names:

        res[name]= {}

        exp_path = fr".\baseline\experts\{ds_name}\{name}.pkl"
        trained_exp = joblib.load(exp_path)

        experts_obj[name]= trained_exp
        
        baseline_inst= Baseline(user=experts_obj[name],
                                label=DATASETS[ds_name]['target'],
                                train_set=dfs['hic_train'],
                                test_set=dfs['mic'],
                                val_set=dfs['calibration'],
                                x_train=X_hic_train,
                                y_train=y_hic_train,
                                x_test=X_mic,
                                y_test=y_mic,
                                x_val=X_val,
                                y_val=y_val
                                )
        
        train, test, val = baseline_inst.fit_expert(scaler)

        print(f"Train Predictions Unique Values: {train['expert prediction'].unique()}")
        
        print(f"Test Predictions Unique Values: {test['expert prediction'].unique()}")

        print(f"Val Predictions Unique Values: {val['expert prediction'].unique()}")

        res[name]['train']= train
        res[name]['test']= test
        res[name]['val']= val

        # saving stuff
        train_path= fr".\baseline\mic\labeled_ds\{ds_name}\train"
        os.makedirs(train_path, exist_ok=True)
        file_path = os.path.join(train_path, f"train_baseline_{name}.csv")

        train.to_csv(file_path, index=False)

        test_path= fr".\baseline\mic\labeled_ds\{ds_name}\test"
        os.makedirs(test_path, exist_ok=True)
        file_path = os.path.join(test_path, f"test_baseline_{name}.csv")

        test.to_csv(file_path, index=False)


        val_path= fr".\baseline\mic\labeled_ds\{ds_name}\val"
        os.makedirs(val_path, exist_ok=True)
        file_path = os.path.join(val_path, f"val_baseline_{name}.csv")

        val.to_csv(file_path, index=False)

        #training nets straight away
        
        
        layers = NET_CONFIGS['small']['layers']
        step_size= NET_CONFIGS['small']['step_size']
        gamma= NET_CONFIGS['small']['gamma']

        net_calibration(ds_name= ds_name,  
                user_suffix= name, 
                iter=None, 
                target= DATASETS[ds_name]['target'] , 
                train_set= train, 
                val_set=val, 
                feature_order=base_params['feature_order'], #without label 
                net_layers=layers, 
                net_params=DATASETS[ds_name]['net_params'], 
                step_size=step_size,
                gamma=gamma,
                device=device,
                baseline= True)
        
        def_net_path= fr".\baseline\mic\two_stage_deferral\nets\{ds_name}"

                 run_calibration(def_net_path=def_net_path, # DA PATH
                    df_switch= train, # questo deve avere le machine prediction btw
                    ds_name=ds_name,
                    params=params,
                    device=device,
                    iteration,
                    def_net_layers, # you get it from outside since you loop outside,
                    batch_size=128, #DEF VALUE, CAN REMOVE
                    epochs= 20,
                    baseline= False
                    ):


        

100%|██████████| 1083/1083 [00:00<00:00, 9357.39it/s]


Train Predictions Unique Values: [1 0]
Test Predictions Unique Values: [1 0]
Val Predictions Unique Values: [0 1]
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001


2026-03-23 21:46:57,512 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.69
End of Epoch 4: Learning Rate 0.001
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.44 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.44 Avg loss: 0.69
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.58 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.58 Avg loss: 0.69
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.61 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.61 Avg loss: 0.68
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.60 Avg loss: 0.68
Validation Results - Epoch[4] Avg accuracy: 0.61 Avg loss: 0.68
End of Epoch 4: Learning R

2026-03-23 21:47:00,297 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

Train Predictions Unique Values: [0 1]
Test Predictions Unique Values: [0 1]
Val Predictions Unique Values: [1 0]
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.70


2026-03-23 21:47:09,945 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.69
End of Epoch 4: Learning Rate 0.001
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.44 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.44 Avg loss: 0.69
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.58 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.58 Avg loss: 0.69
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.61 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.61 Avg loss: 0.68
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.60 Avg loss: 0.68
Validation Results - Epoch[4] Avg accuracy: 0.61 Avg loss: 0.68
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.62 Avg l

2026-03-23 21:47:12,753 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[15] Avg accuracy: 0.63 Avg loss: 0.64
Validation Results - Epoch[15] Avg accuracy: 0.64 Avg loss: 0.64
End of Epoch 15: Learning Rate 0.0008


In [ ]:
base_params= {
    "cats": categoricals, #lst
    "num": numericals, #lss
    "warm_up_set": df_avv.copy(),
    "batch1": df_batch_1.copy(),
    "batch1_test": test_batch_1.copy(),
    "batch3":dfs['mic'].copy(),
    "validation_set": dfs['calibration'].copy(), #obtained by hic
    "feature_order": initial_ordering.copy(), #without the label ?
    "incremental_learner_name":"ARF",
    "n_cols": len(initial_ordering), #obtained by hic
    "mic_model_name": "Def_Net",
    "strat_1_name": "Confidence",
    "strat_2_name": "Mao",
    
}


learner_path= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
path= os.path.join(learner_path, f"{base_params["incremental_learner_name"]}_model.pkl")
trained_learner= joblib.load(path)

prepr_path= DATASETS[ds_name]['base_obj_paths']['preprocessors']
path= os.path.join(prepr_path, f"{base_params["incremental_learner_name"]}_preprocessor.pkl")
base_prepr= joblib.load(path)

base_objects = {
      "preprocessor": copy.deepcopy(base_prepr), #or take from path
      "incremental_learner":copy.deepcopy(trained_learner), # or take from path
      "scaler": None
} 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]

batch3_path= DATASETS[ds_name]['paths']['scaled_mic_batch']
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
mic_save_path = DATASETS[ds_name]['paths']['mic_df_save_path']



In [ ]:
layers= [32,16]

for name, exp_object in res.items():
    run_mic(ds_name=ds_name,device=device, layers=layers, params=)

#### Tests

In [ ]:
def get_l2d_results_and_metrics(method, model_fn, n_classes, 
                                 test_data,
                                device, path):
  
    """method (str): L2D algorithm to use. Either of: {'MixtureOfExperts', 'CrossEntropy', 'OneVsAll', 'Realizable', 'CompareConfidence', 'DifferentiableTriage', 'SelectivePrediction'}
        model_fn: name of function to initialize the nn model
        n_classes (int): cardinality of label space
    """

    print('Computing L2D algorithm with method: ', method)
    print('Initializing the neural network model(s)')
    # initialize the one or two nn
    if method in ['MixtureOfExperts', 'CrossEntropy', 'OneVsAll', 'Realizable']:
        model_1 = model_fn(n_classes+1, device, path)
    elif method == 'SelectivePrediction':
        model_1 = model_fn(n_classes, device, path)
    elif method in ['CompareConfidence','DifferentiableTriage']:
        model_1 = model_fn(n_classes, device, path) # classifier
        model_2 = model_fn(2, device, path) # rejector
    else:
        print('Unknown method: ', method)
        return

    print('Initializing the L2D algorithm')
    if method == 'MixtureOfExperts':
        l2d_algo = MixtureOfExperts(model_1, device)
    elif method == 'CrossEntropy':
        l2d_algo = LceSurrogate(1, 300, model_1, device)
    elif method == 'OneVsAll':
        l2d_algo = OVASurrogate(1, 300, model_1, device)
    elif method == 'Realizable':
        l2d_algo = RealizableSurrogate(1, 300, model_1, device)
    elif method == 'CompareConfidence':
        l2d_algo = CompareConfidence(model_1, model_2, device)
    elif method == 'DifferentiableTriage':
        l2d_algo = DifferentiableTriage(model_1, model_2, device, 0.000, "human_error")
    elif method == 'SelectivePrediction':
        l2d_algo = SelectivePrediction(model_1, device)
    else:
        print('Unknown method: ', method)
        return

    # retrieving test loader
    _, _, test_loader= create_loader(test_data, target=DATASETS[ds_name]['target'], features=base_params['feature_order'])
   
    #retrieving net
    

    print('Computing predictions on the test set and corresponding metrics')
    # compute prediction on the test dataset
    l2d_test = l2d_algo.test(test_loader)

    # compute deferral metrics
    l2d_def_metrics = compute_deferral_metrics(l2d_test)

    # compute classifier metrics
    l2d_clf_metrics = compute_classification_metrics(l2d_test)

    # compute_deferral_metrics(data_test_modified) on different coverage levels, first element of list is compute_deferral_metrics(data_test)
    l2d_coverage_metrics = compute_coverage_v_acc_curve(l2d_test)

    return l2d_algo, l2d_test, l2d_def_metrics, l2d_clf_metrics, l2d_coverage_metrics